# Phase 1 — record thresholds (1979–2015) and 2020 records

Streams ERA5 from Google Cloud (Colab sits next to it, so reading is fast and free) and writes
small result files to your Google Drive.

**Runtime:** a normal **CPU** runtime is enough — this phase only reads data (Runtime → Change
runtime type → CPU). The long step (§6) saves each finished year, so after a disconnect just
reconnect, run §1–§3 again and rerun §6: it continues where it stopped.

## 1. Get the code

In [ ]:
import os, subprocess
REPO_URL = "https://github.com/RomanGorelsky/graphcast-extremes"
if os.path.exists("/content/graphcast-extremes"):
    subprocess.run(["git", "-C", "/content/graphcast-extremes", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/graphcast-extremes"], check=True)
%cd /content/graphcast-extremes
!git log -1 --format="%h %s"

## 2. Install (≈3 min; CPU-only is fine here)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. Storage

Results (including the per-year files that make §6 resumable) go to Drive; the model's example
file (needed for the land-sea mask and the regridding check) goes to the local disk.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.environ["EXTREMES_DATA"] = "/content/data"
os.environ["EXTREMES_RESULTS"] = "/content/drive/MyDrive/graphcast-extremes/results"
os.makedirs(os.environ["EXTREMES_RESULTS"], exist_ok=True)
!python scripts/00_download_assets.py
!python -m pytest -q tests

## 4. What is in the ERA5 bucket? (already checked - optional)

In [ ]:
!python scripts/10_check_regridding.py --list

## 5. Which ERA5 source matches the model's 1° data?

Compares WeatherBench2's ready-made 1° store and two ways of converting the 0.25° store with the
authors' 1° example. Must end with `exact match`; otherwise stop and send the output to Claude.

In [ ]:
!python scripts/10_check_regridding.py

## 6. Record thresholds 1979–2015

First one year as a timing test, then all 37 years (rerun the second cell after any disconnect —
finished years are skipped). Multiply the one-year time by 37 for the total.

In [ ]:
!python scripts/11_record_thresholds.py --years 1979 1979

In [ ]:
!python scripts/11_record_thresholds.py

## 7. Records in 2020

In [ ]:
!python scripts/12_find_records.py --year 2020

## 8. Sanity maps

In [ ]:
!python scripts/13_sanity_maps.py --year 2020
from IPython.display import Image, display
display(Image(os.environ["EXTREMES_RESULTS"] + "/phase1/records_2020_maps.png"))

## 9. Bring the results to the Mac

Download from Drive (`MyDrive/graphcast-extremes/results/phase1/`) into
`~/Documents/GitHub/graphcast-extremes/results/phase1/`:
`thresholds_1979-2015_1deg.nc`, `records_2020.csv.gz`, `records_2020_summary.json`,
`records_2020_maps.png`, `regrid_check.json`. (The `yearly/` folder can stay on Drive.)